# ディープ コンボリューショナル敵対的生成ネットワーク（DCGAN）

**学習目標**

- Keras で GAN アーキテクチャ（ジェネレータとディスクリミネータで構成）を構築する
- ジェネレータとディスクリミネータの損失を定義する
- `tf.GradientTape()` と `@tf.function` を使用して GAN のトレーニングステップを定義する
- MNIST データセットで GAN をトレーニングする

## はじめに
このノートブックでは、[ディープ コンボリューショナル敵対的生成ネットワーク](https://arxiv.org/pdf/1511.06434.pdf)（DCGAN: Deep Convolutional Generative Adversarial Network）を使用して、手書き数字の画像を生成する[敵対的生成ネットワーク](https://arxiv.org/abs/1406.2661)（GAN: Generative Adversarial Network）を構築およびトレーニングする方法を説明します。

GAN は、敵対的プロセスを通じて同時にトレーニングされる 2 つのモデルで構成されます。*ジェネレータ（生成器 / 「芸術家」）*は本物に見える画像を作成することを学習し、*ディスクリミネータ（識別器 / 「美術評論家」）*は本物の画像と偽物の画像を見分けることを学習します。

![ジェネレータとディスクリミネータの図](../assets/gan1.png)

トレーニング中、*ジェネレータ*は本物そっくりの画像を作成するのが次第に上手になり、一方*ディスクリミネータ*は偽物の画像を見分けるのが上手になります。このプロセスは、*ディスクリミネータ*が本物の画像と偽物の画像を区別できなくなったときに均衡に達します。

![ジェネレータとディスクリミネータの2つ目の図](../assets/gan2.png)

このノートブックでは、MNIST の数字を生成する GAN を構築します。このノートブックでは MNIST データセットを使ってこのプロセスを示します。次のアニメーションは、50 エポックにわたってトレーニングされた*ジェネレータ*によって生成された一連の画像を示しています。画像はランダムノイズから始まり、時間の経過とともに徐々に手書き数字らしくなっていきます。

![サンプル出力](https://tensorflow.org/images/gan/dcgan.gif)

## Keras とその他のライブラリのインポート

In [ ]:
import os
import warnings

os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

In [ ]:
import glob
import os
import time

import imageio
import keras
import matplotlib.pyplot as plt
import numpy as np
import PIL
import tensorflow as tf
from IPython import display
from keras import layers

## データセットの読み込みと準備

このノートブックでは、ジェネレータとディスクリミネータのトレーニングに MNIST データセットを使用します。ジェネレータは、MNIST データに類似した手書き数字を生成します。

In [ ]:
(train_images, train_labels), (_, _) = keras.datasets.mnist.load_data()

In [ ]:
train_images = train_images.reshape(train_images.shape[0], 28, 28, 1).astype(
    "float32"
)
train_images = (train_images - 127.5) / 127.5  # 画像を [-1, 1] に正規化


In [ ]:
BUFFER_SIZE = 60000
BATCH_SIZE = 256

次に、`tf.data` を使用して入力パイプラインを定義します。以下のパイプラインは、`train_images` をテンソルスライスとして読み込み、トレーニング用にサンプルをシャッフルおよびバッチ処理します。

In [ ]:
# データをバッチ処理およびシャッフル
train_dataset = tf.data.Dataset.from_tensor_slices(train_images)
train_dataset = train_dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE)


## ジェネレータとディスクリミネータのモデルを作成する

ジェネレータとディスクリミネータのモデルは両方とも、[Keras Sequential API](https://keras.io/guides/sequential_model/) を使用して定義します。

### ジェネレータ

ジェネレータは、[keras.layers.Conv2DTranspose](https://keras.io/api/layers/convolution_layers/convolution2d_transpose/)（アップサンプリング）レイヤを使用して、シード（ランダムノイズ）から画像を生成します。このシードを入力として受け取る `Dense` レイヤから開始し、目的の画像サイズである 28x28x1 に達するまで複数回アップサンプリングを行います。

**演習:** 下のコードを完成させてジェネレータモデルを作成してください。ランダムノイズを入力として受け取る Dense レイヤから開始します。ランダムノイズは `keras.random.normal([1, 100])` を使用して作成します。複数のレイヤにわたって `keras.layers.Conv2DTranspose` を使用し、100 次元のランダムノイズを最終的に 28x28x1 の次元（元の MNIST 数字の形状）にアップサンプリングします。

ヒント: `BatchNormalization` や `LeakyReLU` などのさまざまな活性化関数を試してみてください。

In [ ]:
# TODO 1


def make_generator_model():
    model = keras.Sequential()

    # TODO: ここにコードを記述してください。

    assert model.output_shape == (None, 28, 28, 1)

    return model


（まだトレーニングされていない）ジェネレータを使用して画像を作成してみましょう。

In [ ]:
generator = make_generator_model()

noise = keras.random.normal([1, 100])
generated_image = generator(noise, training=False)

plt.imshow(generated_image[0, :, :, 0], cmap="gray")

`.summary()` を使用して、ジェネレータとディスクリミネータのモデルの全体概要を確認できます。

In [ ]:
generator.summary()

In [ ]:
keras.utils.plot_model(
    generator, show_shapes=True, show_layer_names=True, dpi=75
)

### ディスクリミネータ

次に、ディスクリミネータを構築します。ディスクリミネータは CNN ベースの画像分類器です。28x28x1 の形状の画像を受け取り、その画像が本物であるかどうかを示す単一の分類結果を返す必要があります。

**演習:** 下のコードを完成させて CNN ベースのディスクリミネータモデルを作成してください。モデルは 28x28x1 の形状のテンソルを入力として受け取る二値分類器である必要があります。畳み込み、活性化関数、ドロップアウトのさまざまな組み合わせを試してみてください。

In [ ]:
# TODO 1.


def make_discriminator_model():
    model = keras.Sequential()

    # TODO: ここにコードを記述してください。

    assert model.output_shape == (None, 1)

    return model


（まだトレーニングされていない）ディスクリミネータを使用して、生成された画像が本物か偽物かを分類してみましょう。モデルは本物の画像に対して正の値を、偽物の画像に対して負の値を出力するようにトレーニングされます。

In [ ]:
discriminator = make_discriminator_model()
decision = discriminator(generated_image)
print(decision)

In [ ]:
discriminator.summary()

In [ ]:
keras.utils.plot_model(
    discriminator, show_shapes=True, show_layer_names=True, dpi=75
)

## 損失とオプティマイザの定義

次に、ジェネレータモデルとディスクリミネータモデルの両方の損失関数とオプティマイザを定義します。ジェネレータとディスクリミネータはいずれも `BinaryCrossentropy` を使用します。

ディスクリミネータモデルはロジット（sigmoid レイヤからの確率ではなく）を返すため、`from_logits` を `True` に設定する必要があります。

In [ ]:
# このメソッドはクロスエントロピー損失を計算するヘルパー関数を返します
cross_entropy = keras.losses.BinaryCrossentropy(from_logits=True)


### ディスクリミネータの損失

以下のメソッドは、ディスクリミネータが本物の画像と偽物の画像をどれだけ適切に識別できているかを定量化します。

ディスクリミネータをトレーニングする際（つまりジェネレータを固定する場合）、損失関数には 2 つの部分があることを思い出してください。本物のデータからサンプリングした場合の損失と、偽物のデータからサンプリングした場合の損失です。以下の関数は、本物の画像に対するディスクリミネータの予測と 1 の配列を比較し、偽物（生成された）の画像に対するディスクリミネータの予測と 0 の配列を比較します。

**演習:**
以下のメソッドのコードを完成させてください。`real_loss` は本物の画像に対するディスクリミネータの予測のクロスエントロピーを返し、`fake_loss` は偽物の画像に対するディスクリミネータの予測のクロスエントロピーを返す必要があります。

In [ ]:
#TODO 2
def discriminator_loss(real_output, fake_output):
    real_loss = # TODO: ここにコードを記述してください。
    fake_loss = # TODO: ここにコードを記述してください。
    total_loss = # TODO: ここにコードを記述してください。
    return total_loss


### ジェネレータの損失
ジェネレータの損失は、ディスクリミネータをどれだけうまく騙すことができたかを定量化します。直感的には、ジェネレータがうまく機能していれば、ディスクリミネータは偽物の画像を本物（つまり 1）として分類します。ここでは、生成された画像に対するディスクリミネータの判定と 1 の配列を比較します。

**演習:**
ジェネレータの出力のクロスエントロピー損失を返すようにコードを完成させてください。

In [ ]:
# TODO 2


def generator_loss(fake_output):
    return  # ここにコードを記述してください。


### ジェネレータとディスクリミネータのオプティマイザ

2 つのネットワークを個別にトレーニングするため、ディスクリミネータとジェネレータそれぞれに対して 2 つの独立したオプティマイザを定義する必要があることに注意してください。

In [ ]:
generator_optimizer = keras.optimizers.Adam(1e-4)
discriminator_optimizer = keras.optimizers.Adam(1e-4)

### チェックポイントの保存
このノートブックでは、長時間のトレーニングタスクが中断された場合に役立つ、モデルの保存と復元の方法も示します。

In [ ]:
checkpoint_dir = "./gan_training_checkpoints"
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
checkpoint = tf.train.Checkpoint(
    generator_optimizer=generator_optimizer,
    discriminator_optimizer=discriminator_optimizer,
    generator=generator,
    discriminator=discriminator,
)

## トレーニングループの定義

次に、GAN をトレーニングするためのトレーニングループを定義します。以下では、トレーニング用のグローバル変数を設定します。

In [ ]:
EPOCHS = 50
NOISE_DIM = 100
NUM_EXPAMPLES_TO_GENERATE = 16

# アニメーション GIF で進捗を視覚化しやすくするため、
# このシードを経時的に再利用します
seed = keras.random.normal([NUM_EXPAMPLES_TO_GENERATE, NOISE_DIM])


トレーニングループは、ジェネレータが入力をとしてランダムシードを受け取ることから始まります。そのシードを使用して画像が生成されます。次に、ディスクリミネータを使用して、本物の画像（トレーニングセットから取得）と偽物の画像（ジェネレータによって生成）を分類します。これらのモデルごとに損失が計算され、その勾配を使用してジェネレータとディスクリミネータが更新されます。

**演習:**
下のコードを完成させて、GAN のトレーニングループを定義してください。以下の `tf.function` の使用に注目してください。このアノテーションにより、関数 `train_step` が「コンパイル」されます。`train_step` 関数は入力として画像のバッチを受け取ります。関数の残りの部分では、
- `noise` を入力として `generator` 関数を使用して `generated_images` を作成します
- `images` と `generated_images` にディスクリミネータモデルを適用して、それぞれ `real_output` と `fake_output` を作成します
- 上で定義したメソッドを使用して `gen_loss` と `disc_loss` を定義します
- `gen_tape` と `disc_tape` を使用して、それぞれジェネレータとディスクリミネータの勾配を計算します

最後に、`.apply_gradients` メソッドを使用して、`generator_optimizer` と `discriminator_optimizer` の勾配ステップを実行します。

In [ ]:
# TODO 3
@tf.function
def train_step(images):
    noise = keras.random.normal([BATCH_SIZE, NOISE_DIM])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = # TODO: ここにコードを記述してください。

        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)

        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    gradients_of_generator = gen_tape.gradient(
        gen_loss, generator.trainable_variables
    )
    gradients_of_discriminator = disc_tape.gradient(
        disc_loss, discriminator.trainable_variables
    )

    generator_optimizer.apply_gradients(
        zip(gradients_of_generator, generator.trainable_variables)
    )
    discriminator_optimizer.apply_gradients(
        zip(gradients_of_discriminator, discriminator.trainable_variables)
    )


上記の `train_step` 関数を使用して、GAN のトレーニングを定義します。ここで、`train` 関数は引数として `tf.data` データセットとトレーニングのエポック数を受け取ることに注意してください。

In [ ]:
def train(dataset, epochs):
    for epoch in range(epochs):
        start = time.time()

        for image_batch in dataset:
            train_step(image_batch)

        # 進捗に合わせて GIF 用の画像を生成
        display.clear_output(wait=True)
        generate_and_save_images(generator, epoch + 1, seed)

        # 15 エポックごとにモデルを保存
        if (epoch + 1) % 15 == 0:
            checkpoint.save(file_prefix=checkpoint_prefix)

        print(f"Time for epoch {epoch + 1} is {time.time() - start} sec")

    # 最終エポック後に生成
    display.clear_output(wait=True)
    generate_and_save_images(generator, epochs, seed)


**画像の生成と保存**
画像を生成して保存するための小さなヘルパー関数を使用します。

In [ ]:
def generate_and_save_images(model, epoch, test_input):
    # training が False に設定されていることに注意してください。
    # これにより、すべてのレイヤが推論モード（batchnorm）で実行されます。
    predictions = model(test_input, training=False)

    fig = plt.figure(figsize=(4, 4))

    for i in range(predictions.shape[0]):
        plt.subplot(4, 4, i + 1)
        plt.imshow(predictions[i, :, :, 0] * 127.5 + 127.5, cmap="gray")
        plt.axis("off")

    plt.savefig(f"./gan_images/image_at_epoch_{epoch:04d}.png")
    plt.show()


## モデルのトレーニング
上で定義した `train()` メソッドを呼び出して、ジェネレータとディスクリミネータを同時にトレーニングします。GAN のトレーニングには注意が必要です。ジェネレータとディスクリミネータがお互いを圧倒しないこと（例えば、同等のペースでトレーニングされること）が重要です。

トレーニングの開始時には、生成された画像はランダムノイズのように見えます。トレーニングが進むにつれて、生成された数字は徐々に本物らしくなっていきます。約 50 エポック後には、MNIST の数字に類似するようになります。これには 1 エポックあたり約 1〜2 分かかる場合があります。

In [ ]:
!test -d ./gan_images || mkdir ./gan_images/

In [ ]:
# TODO 4
# TODO: ここにコードを記述してください。


## GIF の作成

最後に、トレーニングを通じて生成された画像の変遷を示す GIF を作成します。

In [ ]:
# エポック番号を使用して 1 枚の画像を表示


def display_image(epoch_no):
    return PIL.Image.open(f"./gan_images/image_at_epoch_{epoch_no:04d}.png")


In [ ]:
display_image(EPOCHS)

`imageio` を使用して、トレーニング中に保存された画像からアニメーション GIF を作成します。

In [ ]:
anim_file = "dcgan.gif"

with imageio.get_writer(anim_file, mode="I") as writer:
    filenames = glob.glob("./gan_images/image*.png")
    filenames = sorted(filenames)
    last = -1
    for i, filename in enumerate(filenames):
        frame = 2 * (i**0.5)
        if round(frame) > round(last):
            last = frame
        else:
            continue
        image = imageio.imread(filename)
        writer.append_data(image)
    image = imageio.imread(filename)
    writer.append_data(image)

display.Image(filename=anim_file)

## 次のステップ

このチュートリアルでは、GAN を記述してトレーニングするために必要な完全なコードを示しました。次のステップとして、別のデータセット、例えば [Kaggle で利用可能](https://www.kaggle.com/jessicali9530/celeba-dataset) な Large-scale Celeb Faces Attributes（CelebA）データセットなどを試してみるとよいでしょう。GAN の詳細については、[NIPS 2016 Tutorial: Generative Adversarial Networks](https://arxiv.org/abs/1701.00160) を参照することをお勧めします。

Copyright 2020 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License